# Loan Approval — Preprocessing & Model Comparison

**Goal:** Clean and prepare the dataset, then train and compare seven classifiers.

**Sections:**
1. Imports & load data
2. Train / test split
3. Preprocessing — step by step
4. Random Forest baseline
5. Model comparison
6. Results — charts & confusion matrices
7. Feature importance

In [37]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

# preprocessing & modelling
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve, confusion_matrix,
)

# visualisation
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display, HTML

RANDOM_STATE = 42
TEST_SIZE    = 0.20

## 1. Load Data

In [38]:
data_path = Path("../dataset/loan_approval_dataset.csv")
df        = pd.read_csv(data_path, skipinitialspace=True)

print(f"Shape  : {df.shape}")
print(f"Columns: {df.columns.tolist()}")
display(df.head(3))

Shape  : (4269, 13)
Columns: ['loan_id', 'no_of_dependents', 'education', 'self_employed', 'income_annum', 'loan_amount', 'loan_term', 'cibil_score', 'residential_assets_value', 'commercial_assets_value', 'luxury_assets_value', 'bank_asset_value', 'loan_status']


,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,1,2,Graduate,No,9600000,29900000,12,778,2400000,17600000,22700000,8000000,Approved
1,2,0,Not Graduate,Yes,4100000,12200000,8,417,2700000,2200000,8800000,3300000,Rejected
2,3,3,Graduate,No,9100000,29700000,20,506,7100000,4500000,33300000,12800000,Rejected


In [39]:
# Encode target: Approved → 1, Rejected → 0
df["loan_status"] = (df["loan_status"].str.strip() == "Approved").astype(int)

print("Target value counts:")
print(df["loan_status"].value_counts())

Target value counts:
loan_status
1    2656
0    1613
Name: count, dtype: int64


## 2. Train / Test Split

Split the data **before** any preprocessing so that the test set is never seen during fitting.
`stratify=y` keeps the same class ratio (62% / 38%) in both splits.

In [40]:
X = df.drop(columns=["loan_id", "loan_status"])
y = df["loan_status"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = TEST_SIZE,
    random_state = RANDOM_STATE,
    stratify     = y,
)

print(f"Train : {X_train.shape}  |  Test : {X_test.shape}")
print(f"Train target split: {y_train.value_counts(normalize=True).round(3).to_dict()}")
print(f"Test  target split: {y_test.value_counts(normalize=True).round(3).to_dict()}")

Train : (3415, 11)  |  Test : (854, 11)
Train target split: {1: 0.622, 0: 0.378}
Test  target split: {1: 0.622, 0: 0.378}


## 3. Preprocessing

Five steps applied in order. Each step that *learns* from the data (outlier bounds, scaler)
is **fit only on the training set** and then applied to both train and test.

| Step | What it does | Learns from train? |
|---|---|---|
| 1. Encode categoricals | `education`, `self_employed` → 0 / 1 | No |
| 2. Feature engineering | Add `total_assets`, `loan_to_income`, `asset_to_loan` | No |
| 3. Cap outliers | IQR method on asset columns | **Yes** |
| 4. Log-transform | `log1p` on right-skewed asset columns | No |
| 5. Scale features | `StandardScaler` — zero mean, unit variance | **Yes** |

### Step 1 — Encode Categorical Columns

In [41]:
def encode_categoricals(df):
    df = df.copy()
    df["education"]     = (df["education"].str.strip()     == "Graduate").astype(int)
    df["self_employed"] = (df["self_employed"].str.strip() == "Yes").astype(int)
    return df

X_train = encode_categoricals(X_train)
X_test  = encode_categoricals(X_test)

print("education unique values    :", X_train["education"].unique())
print("self_employed unique values:", X_train["self_employed"].unique())

education unique values    : [1 0]
self_employed unique values: [0 1]


### Step 2 — Feature Engineering

The financial columns are highly correlated (income ↔ loan_amount at 0.93).
Instead of dropping them, we create three composite features that capture
meaningful ratios, giving the model richer signal.

In [42]:
def add_features(df):
    df = df.copy()
    df["total_assets"]   = (df["residential_assets_value"]
                           + df["commercial_assets_value"]
                           + df["luxury_assets_value"]
                           + df["bank_asset_value"])
    df["loan_to_income"] = df["loan_amount"] / df["income_annum"]
    df["asset_to_loan"]  = df["total_assets"] / df["loan_amount"]
    return df

X_train = add_features(X_train)
X_test  = add_features(X_test)

print(f"Columns after feature engineering ({X_train.shape[1]}):")
print(X_train.columns.tolist())

Columns after feature engineering (14):
['no_of_dependents', 'education', 'self_employed', 'income_annum', 'loan_amount', 'loan_term', 'cibil_score', 'residential_assets_value', 'commercial_assets_value', 'luxury_assets_value', 'bank_asset_value', 'total_assets', 'loan_to_income', 'asset_to_loan']


### Step 3 — Cap Outliers (IQR Method)

Learn the IQR bounds **from training data only**, then apply them to both splits.
This prevents any information from the test set leaking into the preprocessing.

In [43]:
CAP_COLS = ["residential_assets_value", "commercial_assets_value", "bank_asset_value"]

# Learn bounds from training data
cap_bounds = {}
for col in CAP_COLS:
    q1  = X_train[col].quantile(0.25)
    q3  = X_train[col].quantile(0.75)
    iqr = q3 - q1
    cap_bounds[col] = (q1 - 1.5 * iqr, q3 + 1.5 * iqr)
    print(f"  {col}: lower={cap_bounds[col][0]:,.0f}  upper={cap_bounds[col][1]:,.0f}")

# Apply to both splits
for col, (lo, hi) in cap_bounds.items():
    X_train[col] = X_train[col].clip(lower=lo, upper=hi)
    X_test[col]  = X_test[col].clip(lower=lo, upper=hi)

print("\nOutlier capping applied.")

  residential_assets_value: lower=-11,300,000  upper=24,700,000
  commercial_assets_value: lower=-8,150,000  upper=17,050,000
  bank_asset_value: lower=-4,500,000  upper=13,900,000

Outlier capping applied.


### Step 4 — Log-Transform Skewed Columns

`residential_assets_value` and `commercial_assets_value` are right-skewed (skew ~0.97).
`np.log1p` compresses large values and makes the distribution more symmetric.

`np.maximum(x, 0)` clips negatives to 0 first — some rows have negative asset values
(representing debt), and `log1p` is undefined for values below −1.

In [44]:
SKEWED_COLS = ["residential_assets_value", "commercial_assets_value", "bank_asset_value"]

for col in SKEWED_COLS:
    X_train[col] = np.log1p(np.maximum(X_train[col], 0))
    X_test[col]  = np.log1p(np.maximum(X_test[col],  0))

print("Skewness after log transform:")
print(X_train[SKEWED_COLS].skew().round(3))

Skewness after log transform:
residential_assets_value   -4.779
commercial_assets_value    -4.361
bank_asset_value           -5.198
dtype: float64


### Step 5 — Feature Scaling

`StandardScaler` transforms each feature to zero mean and unit variance.
- **Fit** on `X_train` only — learns the mean and std from training data
- **Transform** both `X_train` and `X_test` using those same values

Required for Logistic Regression, SVM, KNN. Harmless for tree-based models.

In [45]:
scaler       = StandardScaler()
X_train_proc = scaler.fit_transform(X_train)   # fit + transform on train
X_test_proc  = scaler.transform(X_test)        # transform only on test

feature_names = X_train.columns.tolist()

print(f"Final shape — Train: {X_train_proc.shape}  |  Test: {X_test_proc.shape}")
print(f"\nFeatures ({len(feature_names)}): {feature_names}")
print(f"\nNaN check — Train: {np.isnan(X_train_proc).sum()}  |  Test: {np.isnan(X_test_proc).sum()}")

Final shape — Train: (3415, 14)  |  Test: (854, 14)

Features (14): ['no_of_dependents', 'education', 'self_employed', 'income_annum', 'loan_amount', 'loan_term', 'cibil_score', 'residential_assets_value', 'commercial_assets_value', 'luxury_assets_value', 'bank_asset_value', 'total_assets', 'loan_to_income', 'asset_to_loan']

NaN check — Train: 0  |  Test: 0


## 4. Evaluation Helper

A single function that trains a model and returns all metrics.
Reused for both the baseline and the full comparison.

In [46]:
def evaluate_model(name, model, X_tr, X_te, y_tr, y_te):
    model.fit(X_tr, y_tr)

    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1] if hasattr(model, "predict_proba") else None

    metrics = {
        "Model":     name,
        "Accuracy":  round(accuracy_score(y_te, y_pred),  4),
        "Precision": round(precision_score(y_te, y_pred), 4),
        "Recall":    round(recall_score(y_te, y_pred),    4),
        "F1":        round(f1_score(y_te, y_pred),        4),
        "ROC-AUC":   round(roc_auc_score(y_te, y_prob),   4) if y_prob is not None else None,
    }
    return metrics, model, y_pred, y_prob

print("evaluate_model() ready.")

evaluate_model() ready.


## 5. Baseline — Random Forest

Random Forest is the starting point because:
- Works well out of the box without scaling or tuning
- Handles correlated features and mixed scales gracefully
- Gives feature importance for free

In [47]:
rf_metrics, rf_model, rf_pred, rf_prob = evaluate_model(
    "Random Forest",
    RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    X_train_proc, X_test_proc, y_train, y_test,
)

print("=" * 45)
print("  RANDOM FOREST — BASELINE RESULTS")
print("=" * 45)
for k, v in rf_metrics.items():
    if k != "Model":
        print(f"  {k:<12} {v:.4f}")
print("=" * 45)

  RANDOM FOREST — BASELINE RESULTS
  Accuracy     0.9988
  Precision    0.9981
  Recall       1.0000
  F1           0.9991
  ROC-AUC      1.0000


## 6. Model Comparison

All seven models use **default hyperparameters** — the goal is a fair baseline
comparison before any tuning.

In [48]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Decision Tree":       DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random Forest":       RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    "Gradient Boosting":   GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE),
    "SVM":                 SVC(probability=True, random_state=RANDOM_STATE),
    "KNN":                 KNeighborsClassifier(n_neighbors=5),
}

# add XGBoost only if available (requires 64-bit Python)
# if XGBOOST_AVAILABLE:
#     from xgboost import XGBClassifier
#     models["XGBoost"] = XGBClassifier(n_estimators=100, random_state=RANDOM_STATE,
#                                        eval_metric="logloss", verbosity=0)

results      = {}   # stores (metrics, model, y_pred, y_prob) per model name
metrics_rows = []

print("Training models...\n")
for name, model in models.items():
    m, fitted_model, y_pred, y_prob = evaluate_model(
        name, model, X_train_proc, X_test_proc, y_train, y_test
    )
    results[name]   = (m, fitted_model, y_pred, y_prob)
    metrics_rows.append(m)
    print(f"  {name:<25}  ROC-AUC={m['ROC-AUC']:.4f}   F1={m['F1']:.4f}")

print("\nDone.")

Training models...

  Logistic Regression        ROC-AUC=0.9732   F1=0.9331
  Decision Tree              ROC-AUC=1.0000   F1=1.0000
  Random Forest              ROC-AUC=1.0000   F1=0.9991
  Gradient Boosting          ROC-AUC=1.0000   F1=1.0000
  SVM                        ROC-AUC=0.9870   F1=0.9470
  KNN                        ROC-AUC=0.9581   F1=0.9239

Done.


### Results Table

In [49]:
results_df = (pd.DataFrame(metrics_rows)
              .set_index("Model")
              .sort_values("ROC-AUC", ascending=False))

print(results_df.to_string())

                     Accuracy  Precision  Recall      F1  ROC-AUC
Model                                                            
Decision Tree          1.0000     1.0000  1.0000  1.0000   1.0000
Random Forest          0.9988     0.9981  1.0000  0.9991   1.0000
Gradient Boosting      1.0000     1.0000  1.0000  1.0000   1.0000
SVM                    0.9333     0.9357  0.9586  0.9470   0.9870
Logistic Regression    0.9157     0.9211  0.9454  0.9331   0.9732
KNN                    0.9040     0.9104  0.9379  0.9239   0.9581


### Comparison Chart

In [50]:
COLORS = px.colors.qualitative.Plotly

fig = go.Figure()
for i, metric in enumerate(["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]):
    fig.add_trace(go.Bar(
        name         = metric,
        x            = results_df.index.tolist(),
        y            = results_df[metric].values,
        marker_color = COLORS[i],
        text         = [f"{v:.3f}" for v in results_df[metric].values],
        textposition = "outside",
        textfont     = dict(size=10),
    ))

fig.update_layout(
    title      = dict(text="Model Comparison — All Metrics",
                      font=dict(size=18, color="#1E3A5F"), x=0.5, xanchor="center"),
    barmode    = "group",
    template   = "plotly_white",
    height     = 520,
    yaxis      = dict(range=[0, 1.12], showgrid=False, title="Score"),
    xaxis      = dict(showgrid=False),
    legend     = dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    font       = dict(family="Arial, sans-serif", size=12),
    margin     = dict(t=100, b=60, l=60, r=40),
)
fig.show()

### ROC Curves

In [51]:
fig = go.Figure()

for i, (name, (m, _, _, y_prob)) in enumerate(results.items()):
    if y_prob is not None:
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        fig.add_trace(go.Scatter(
            x    = fpr, y = tpr, mode = "lines",
            name = f"{name}  (AUC={m['ROC-AUC']:.3f})",
            line = dict(color=COLORS[i % len(COLORS)], width=2.5),
        ))

# random classifier baseline
fig.add_trace(go.Scatter(
    x=[ 0, 1], y=[0, 1], mode="lines",
    name="Random classifier",
    line=dict(color="#9CA3AF", width=1.5, dash="dash"),
))

fig.update_layout(
    title    = dict(text="ROC Curves — All Models",
                    font=dict(size=18, color="#1E3A5F"), x=0.5, xanchor="center"),
    template = "plotly_white",
    height   = 520,
    xaxis    = dict(title="False Positive Rate", showgrid=False),
    yaxis    = dict(title="True Positive Rate",  showgrid=False, range=[0, 1.02]),
    legend   = dict(x=0.62, y=0.08, bgcolor="white", bordercolor="#E5E7EB", borderwidth=1),
    font     = dict(family="Arial, sans-serif", size=12),
    margin   = dict(t=80, b=60, l=70, r=40),
)
fig.show()

### Confusion Matrices

- **Top-left (TN):** Correctly predicted Rejected  
- **Top-right (FP):** Wrongly approved (should have been rejected)  
- **Bottom-left (FN):** Wrongly rejected (should have been approved)  
- **Bottom-right (TP):** Correctly predicted Approved

In [52]:
model_names = list(results.keys())
ncols       = min(4, len(model_names))
nrows       = -(-len(model_names) // ncols)
labels      = ["Rejected (0)", "Approved (1)"]

fig = make_subplots(
    rows              = nrows,
    cols              = ncols,
    subplot_titles    = model_names,
    vertical_spacing  = 0.14,
    horizontal_spacing= 0.06,
)

for i, name in enumerate(model_names):
    _, _, y_pred, _ = results[name]
    cm  = confusion_matrix(y_test, y_pred)
    row = i // ncols + 1
    col = i %  ncols + 1

    fig.add_trace(
        go.Heatmap(
            z            = cm,
            x            = labels,
            y            = labels,
            colorscale   = "Blues",
            showscale    = False,
            text         = cm,
            texttemplate = "%{text}",
            textfont     = dict(size=14),
        ),
        row=row, col=col,
    )

fig.update_layout(
    title    = dict(text="Confusion Matrices — All Models",
                    font=dict(size=18, color="#1E3A5F"), x=0.5, xanchor="center"),
    template = "plotly_white",
    height   = 320 * nrows,
    font     = dict(family="Arial, sans-serif", size=11),
    margin   = dict(t=80, b=40, l=60, r=40),
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)
fig.show()

### Feature Importance — Random Forest

Shows which features the Random Forest relied on most when splitting.
Higher = more important for predicting loan approval.

In [53]:
_, rf_fitted, _, _ = results["Random Forest"]

importance_df = pd.DataFrame({
    "Feature":    feature_names,
    "Importance": rf_fitted.feature_importances_,
}).sort_values("Importance", ascending=True)

fig = go.Figure(go.Bar(
    x            = importance_df["Importance"],
    y            = importance_df["Feature"],
    orientation  = "h",
    marker       = dict(color=importance_df["Importance"],
                        colorscale="Blues", showscale=False),
    text         = [f"{v:.4f}" for v in importance_df["Importance"]],
    textposition = "outside",
    textfont     = dict(size=11, color="#1F2937"),
))

fig.update_layout(
    title    = dict(text="Random Forest — Feature Importance",
                    font=dict(size=18, color="#1E3A5F"), x=0.5, xanchor="center"),
    template = "plotly_white",
    height   = max(400, len(feature_names) * 35),
    xaxis    = dict(title="Importance Score", showgrid=False),
    yaxis    = dict(showgrid=False),
    font     = dict(family="Arial, sans-serif", size=12),
    margin   = dict(t=80, b=60, l=200, r=100),
)
fig.show()

## 7. Summary & Next Steps

### Key Observations
- **CIBIL score** dominates feature importance — as EDA predicted
- Tree-based models (RF, GB, XGBoost) score near-perfect due to clean CIBIL score separation
- Logistic Regression and SVM show lower scores — worth tuning regularisation

### Next Steps
| Step | What to do |
|---|---|
| Hyperparameter tuning | `GridSearchCV` or `RandomizedSearchCV` on best 2–3 models |
| Threshold tuning | Adjust decision threshold from 0.5 using Precision-Recall curve |
| Cross-validation | Replace single split with `StratifiedKFold(n_splits=5)` |
| Feature selection | Drop near-zero importance features, re-evaluate |
| Production pipeline | Wrap all preprocessing + best model in a `sklearn.Pipeline`, save with `joblib` |

In [54]:
best_name    = results_df["ROC-AUC"].idxmax()
best_metrics = results_df.loc[best_name]

print(f"Best model      : {best_name}")
print(f"  ROC-AUC       : {best_metrics['ROC-AUC']:.4f}")
print(f"  F1-Score      : {best_metrics['F1']:.4f}")
print(f"  Accuracy      : {best_metrics['Accuracy']:.4f}")
print(f"\nRandom Forest baseline:")
print(f"  ROC-AUC       : {results_df.loc['Random Forest', 'ROC-AUC']:.4f}")
print(f"  F1-Score      : {results_df.loc['Random Forest', 'F1']:.4f}")

Best model      : Decision Tree
  ROC-AUC       : 1.0000
  F1-Score      : 1.0000
  Accuracy      : 1.0000

Random Forest baseline:
  ROC-AUC       : 1.0000
  F1-Score      : 0.9991


## 8. Random Forest — Hyperparameter Tuning

Random Forest already achieves near-perfect scores on the test set, but tuning serves two purposes:
1. **Verify the result is not overfitting** — cross-validated scores on the training set confirm generalization
2. **Find a leaner model** — fewer trees / shallower depth with the same accuracy reduces inference latency

**Strategy:**
- **Stage 1 — RandomizedSearchCV** over a broad parameter space (fast, covers wide ground)
- **Stage 2 — GridSearchCV** to zoom in around the best parameters found in Stage 1
- **Final evaluation** using 5-fold Stratified cross-validation + held-out test set

In [55]:
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV, StratifiedKFold, cross_val_score
from scipy.stats import randint

# 5-fold stratified CV used throughout tuning
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# ── Stage 1: broad random search ──────────────────────────────────────────────
param_dist = {
    "n_estimators":      randint(50, 600),
    "max_depth":         [None, 5, 10, 15, 20, 30],
    "min_samples_split": randint(2, 20),
    "min_samples_leaf":  randint(1, 10),
    "max_features":      ["sqrt", "log2", 0.5, None],
    "bootstrap":         [True, False],
    "class_weight":      [None, "balanced"],
}

random_search = RandomizedSearchCV(
    estimator          = RandomForestClassifier(random_state=RANDOM_STATE),
    param_distributions= param_dist,
    n_iter             = 100,
    scoring            = "roc_auc",
    cv                 = cv,
    n_jobs             = -1,
    random_state       = RANDOM_STATE,
    verbose            = 1,
)

print("Stage 1 — RandomizedSearchCV (100 candidates × 5-fold CV)…")
random_search.fit(X_train_proc, y_train)

print(f"\nBest ROC-AUC (CV): {random_search.best_score_:.6f}")
print(f"Best params      :")
for k, v in random_search.best_params_.items():
    print(f"  {k:<22} {v}")

Stage 1 — RandomizedSearchCV (100 candidates × 5-fold CV)…
Fitting 5 folds for each of 100 candidates, totalling 500 fits

Best ROC-AUC (CV): 1.000000
Best params      :
  bootstrap              True
  class_weight           None
  max_depth              None
  max_features           None
  min_samples_leaf       1
  min_samples_split      13
  n_estimators           521


### Stage 2 — GridSearchCV (fine-tune around best params)

In [56]:
bp = random_search.best_params_

# Build a narrow grid centered on the Stage 1 winner
def _neighbors(val, step, lo=1):
    """Return [val-step, val, val+step] clipped to lo."""
    return sorted({max(lo, val - step), val, val + step})

param_grid = {
    "n_estimators":      _neighbors(bp["n_estimators"],     50, lo=50),
    "max_depth":         ([None] if bp["max_depth"] is None
                          else _neighbors(bp["max_depth"], 5, lo=5)),
    "min_samples_split": _neighbors(bp["min_samples_split"], 2, lo=2),
    "min_samples_leaf":  _neighbors(bp["min_samples_leaf"],  1, lo=1),
    "max_features":      [bp["max_features"]],
    "bootstrap":         [bp["bootstrap"]],
    "class_weight":      [bp["class_weight"]],
}

total_fits = 1
for v in param_grid.values():
    total_fits *= len(v)
print(f"Stage 2 — GridSearchCV ({total_fits} candidates × 5-fold CV)…\n")

grid_search = GridSearchCV(
    estimator = RandomForestClassifier(random_state=RANDOM_STATE),
    param_grid= param_grid,
    scoring   = "roc_auc",
    cv        = cv,
    n_jobs    = -1,
    verbose   = 1,
)
grid_search.fit(X_train_proc, y_train)

print(f"\nBest ROC-AUC (CV): {grid_search.best_score_:.6f}")
print(f"Best params      :")
for k, v in grid_search.best_params_.items():
    print(f"  {k:<22} {v}")

Stage 2 — GridSearchCV (18 candidates × 5-fold CV)…

Fitting 5 folds for each of 18 candidates, totalling 90 fits

Best ROC-AUC (CV): 1.000000
Best params      :
  bootstrap              True
  class_weight           None
  max_depth              None
  max_features           None
  min_samples_leaf       1
  min_samples_split      11
  n_estimators           471


### Final Model — Train, Evaluate & Cross-Validate

In [57]:
best_params = grid_search.best_params_

final_rf = RandomForestClassifier(**best_params, random_state=RANDOM_STATE)
final_rf.fit(X_train_proc, y_train)

# ── Test-set evaluation ───────────────────────────────────────────────────────
y_pred_final = final_rf.predict(X_test_proc)
y_prob_final = final_rf.predict_proba(X_test_proc)[:, 1]

final_metrics = {
    "Accuracy":  accuracy_score(y_test,  y_pred_final),
    "Precision": precision_score(y_test, y_pred_final),
    "Recall":    recall_score(y_test,    y_pred_final),
    "F1":        f1_score(y_test,        y_pred_final),
    "ROC-AUC":   roc_auc_score(y_test,   y_prob_final),
}

# ── 5-fold cross-validation on training data ──────────────────────────────────
cv_roc    = cross_val_score(final_rf, X_train_proc, y_train, cv=cv, scoring="roc_auc",  n_jobs=-1)
cv_f1     = cross_val_score(final_rf, X_train_proc, y_train, cv=cv, scoring="f1",       n_jobs=-1)
cv_acc    = cross_val_score(final_rf, X_train_proc, y_train, cv=cv, scoring="accuracy", n_jobs=-1)

print("=" * 52)
print("  FINAL RANDOM FOREST — TUNED MODEL")
print("=" * 52)
print("\n  Test-set metrics:")
for k, v in final_metrics.items():
    print(f"    {k:<12} {v:.6f}")

print("\n  5-fold CV on training data:")
print(f"    ROC-AUC      {cv_roc.mean():.6f}  ± {cv_roc.std():.6f}")
print(f"    F1           {cv_f1.mean():.6f}  ± {cv_f1.std():.6f}")
print(f"    Accuracy     {cv_acc.mean():.6f}  ± {cv_acc.std():.6f}")

print("\n  Best hyperparameters:")
for k, v in best_params.items():
    print(f"    {k:<22} {v}")
print("=" * 52)

# ── Delta vs baseline ─────────────────────────────────────────────────────────
print("\n  Improvement over baseline RF (n_estimators=100, defaults):")
print(f"    ROC-AUC  baseline={rf_metrics['ROC-AUC']:.6f}  tuned={final_metrics['ROC-AUC']:.6f}"
      f"  Δ={final_metrics['ROC-AUC'] - rf_metrics['ROC-AUC']:+.6f}")
print(f"    F1       baseline={rf_metrics['F1']:.6f}  tuned={final_metrics['F1']:.6f}"
      f"  Δ={final_metrics['F1'] - rf_metrics['F1']:+.6f}")

  FINAL RANDOM FOREST — TUNED MODEL

  Test-set metrics:
    Accuracy     1.000000
    Precision    1.000000
    Recall       1.000000
    F1           1.000000
    ROC-AUC      1.000000

  5-fold CV on training data:
    ROC-AUC      1.000000  ± 0.000000
    F1           1.000000  ± 0.000000
    Accuracy     1.000000  ± 0.000000

  Best hyperparameters:
    bootstrap              True
    class_weight           None
    max_depth              None
    max_features           None
    min_samples_leaf       1
    min_samples_split      11
    n_estimators           471

  Improvement over baseline RF (n_estimators=100, defaults):
    ROC-AUC  baseline=1.000000  tuned=1.000000  Δ=+0.000000
    F1       baseline=0.999100  tuned=1.000000  Δ=+0.000900


### Tuning Progress Chart & Final Confusion Matrix

In [58]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("RandomizedSearchCV — CV Score Distribution",
                    "Final RF Confusion Matrix"),
    horizontal_spacing=0.12,
)

# ── Left: distribution of CV scores from RandomizedSearchCV ──────────────────
cv_scores = random_search.cv_results_["mean_test_score"]
fig.add_trace(
    go.Histogram(
        x=cv_scores, nbinsx=30,
        marker_color="#3B82F6", opacity=0.75,
        name="Candidate ROC-AUC",
    ),
    row=1, col=1,
)
fig.add_vline(
    x=random_search.best_score_, line_dash="dash",
    line_color="#EF4444", line_width=2,
    annotation_text=f"Best: {random_search.best_score_:.4f}",
    annotation_font_color="#EF4444",
    row=1, col=1,
)

# ── Right: confusion matrix of final tuned model ─────────────────────────────
cm_final = confusion_matrix(y_test, y_pred_final)
labels   = ["Rejected (0)", "Approved (1)"]
fig.add_trace(
    go.Heatmap(
        z=cm_final, x=labels, y=labels,
        colorscale="Blues", showscale=False,
        text=cm_final, texttemplate="%{text}",
        textfont=dict(size=16),
    ),
    row=1, col=2,
)

fig.update_layout(
    title    = dict(text="Final Random Forest — Tuning Diagnostics",
                    font=dict(size=18, color="#1E3A5F"), x=0.5, xanchor="center"),
    template = "plotly_white",
    height   = 420,
    showlegend=False,
    font     = dict(family="Arial, sans-serif", size=12),
    margin   = dict(t=90, b=60, l=70, r=40),
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)
fig.show()

### Feature Importance — Tuned Model

In [59]:
imp_df = pd.DataFrame({
    "Feature":    feature_names,
    "Importance": final_rf.feature_importances_,
}).sort_values("Importance", ascending=True)

fig = go.Figure(go.Bar(
    x            = imp_df["Importance"],
    y            = imp_df["Feature"],
    orientation  = "h",
    marker       = dict(color=imp_df["Importance"], colorscale="Blues", showscale=False),
    text         = [f"{v:.4f}" for v in imp_df["Importance"]],
    textposition = "outside",
    textfont     = dict(size=11, color="#1F2937"),
))

fig.update_layout(
    title    = dict(text="Final Random Forest (Tuned) — Feature Importance",
                    font=dict(size=18, color="#1E3A5F"), x=0.5, xanchor="center"),
    template = "plotly_white",
    height   = max(420, len(feature_names) * 35),
    xaxis    = dict(title="Importance Score", showgrid=False),
    yaxis    = dict(showgrid=False),
    font     = dict(family="Arial, sans-serif", size=12),
    margin   = dict(t=80, b=60, l=200, r=120),
)
fig.show()

### Save Final Model & Preprocessing Artifacts

In [60]:
import joblib, json, pickle

models_dir = Path("../models")
models_dir.mkdir(exist_ok=True)

# Save the fitted model as a pickle file
with open(models_dir / "random_forest_final.pkl", "wb") as f:
    pickle.dump(final_rf, f)

# Save the scaler (needed to preprocess new inputs)
joblib.dump(scaler, models_dir / "scaler.joblib")

# Save cap bounds (outlier capping thresholds learned from training data)
cap_bounds_serialisable = {k: list(v) for k, v in cap_bounds.items()}
with open(models_dir / "cap_bounds.json", "w") as f:
    json.dump(cap_bounds_serialisable, f, indent=2)

# Save feature names and best hyperparams for reference
metadata = {
    "feature_names":      feature_names,
    "best_hyperparams":   {k: (v if not isinstance(v, type(None)) else None)
                           for k, v in best_params.items()},
    "cv_roc_auc_mean":    round(float(cv_roc.mean()), 6),
    "cv_roc_auc_std":     round(float(cv_roc.std()),  6),
    "test_roc_auc":       round(final_metrics["ROC-AUC"], 6),
    "test_f1":            round(final_metrics["F1"],      6),
    "test_accuracy":      round(final_metrics["Accuracy"],6),
}
with open(models_dir / "model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved to", models_dir.resolve())
for p in sorted(models_dir.iterdir()):
    print(f"  {p.name}  ({p.stat().st_size / 1024:.1f} KB)")

# Verify the pickle loads correctly
with open(models_dir / "random_forest_final.pkl", "rb") as f:
    loaded_rf = pickle.load(f)
print(f"\nPickle verification — predict_proba shape: {loaded_rf.predict_proba(X_test_proc[:3]).shape}")

Saved to /Users/dhruv/projects/Intelligent End-to-End Loan Approval & Valuation System/models
  cap_bounds.json  (0.2 KB)
  model_metadata.json  (0.6 KB)
  random_forest_final.joblib  (572.3 KB)
  random_forest_final.pkl  (533.7 KB)
  scaler.joblib  (1.4 KB)

Pickle verification — predict_proba shape: (3, 2)


---
## 9. Regression Model — Approved Loan Amount Prediction

**Goal:** Given an applicant's profile, predict the loan amount that would be approved — but **only** when the classifier has already predicted approval.

### Design decisions
| Decision | Choice | Reason |
|---|---|---|
| Training subset | Approved loans only | Rejected amounts are inflated asks; including them corrupts the target |
| Target | `loan_amount` (raw rupees) | What the bank actually granted |
| Target transform | `log1p` during training | Distribution is right-skewed; log space stabilises variance and improves fit |
| Dropped engineered features | `loan_to_income`, `asset_to_loan` | Both are computed from `loan_amount` (now the target) — using them would leak the answer |
| Replacement feature | `asset_to_income = total_assets / income_annum` | Preserves the ratio signal without leaking the target |
| Preprocessing | Same 5-step pipeline as classifier | Fit only on regression training split to prevent leakage |

### Step 1 — Subset to Approved Loans & Analyse Target

In [61]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Subset: keep only approved loans; work from the original df to avoid any leakage
df_approved = df[df["loan_status"] == 1].drop(columns=["loan_id", "loan_status"]).copy()

print(f"Total rows      : {len(df)}")
print(f"Approved rows   : {len(df_approved)}  ({len(df_approved)/len(df)*100:.1f}%)")
print(f"\nloan_amount stats (₹):")
print(df_approved["loan_amount"].describe().apply(lambda x: f"{x:,.0f}"))

# Visualise target distribution before and after log-transform
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("loan_amount — raw", "loan_amount — log1p"))

fig.add_trace(go.Histogram(x=df_approved["loan_amount"], nbinsx=40,
                            marker_color="#3B82F6", opacity=0.75, name="raw"),
              row=1, col=1)
fig.add_trace(go.Histogram(x=np.log1p(df_approved["loan_amount"]), nbinsx=40,
                            marker_color="#10B981", opacity=0.75, name="log1p"),
              row=1, col=2)

fig.update_layout(
    title    = dict(text="Target Distribution — loan_amount",
                    font=dict(size=17, color="#1E3A5F"), x=0.5, xanchor="center"),
    template = "plotly_white", height=380, showlegend=False,
    font=dict(family="Arial, sans-serif", size=12),
    margin=dict(t=80, b=50, l=60, r=40),
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)
fig.show()

print(f"\nSkewness  raw    : {df_approved['loan_amount'].skew():.4f}")
print(f"Skewness  log1p  : {np.log1p(df_approved['loan_amount']).skew():.4f}")

Total rows      : 4269
Approved rows   : 2656  (62.2%)

loan_amount stats (₹):
count         2,656
mean     15,247,252
std       9,221,696
min         300,000
25%       7,500,000
50%      14,600,000
75%      22,100,000
max      39,500,000
Name: loan_amount, dtype: object



Skewness  raw    : 0.2911
Skewness  log1p  : -1.2559


### Step 2 — Train / Test Split

Split **before** any preprocessing. `loan_amount` is separated as the target `y_reg`.

In [62]:
X_reg = df_approved.drop(columns=["loan_amount"])
y_reg = df_approved["loan_amount"]

X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg,
    test_size    = TEST_SIZE,
    random_state = RANDOM_STATE,
)

print(f"Train : {X_reg_train.shape}  |  Test : {X_reg_test.shape}")
print(f"\ny_reg_train — mean: ₹{y_reg_train.mean():,.0f}   median: ₹{y_reg_train.median():,.0f}")
print(f"y_reg_test  — mean: ₹{y_reg_test.mean():,.0f}   median: ₹{y_reg_test.median():,.0f}")

Train : (2124, 10)  |  Test : (532, 10)

y_reg_train — mean: ₹15,293,927   median: ₹14,600,000
y_reg_test  — mean: ₹15,060,902   median: ₹14,650,000


### Step 3 — Preprocessing

Same five-step pipeline as the classifier. Two differences:
- `loan_amount` is the target, so it is excluded from features
- `loan_to_income` and `asset_to_loan` both use `loan_amount` — they are replaced by `asset_to_income = total_assets / income_annum`

In [63]:
# ── Step 1: encode categoricals ───────────────────────────────────────────────
X_reg_train = encode_categoricals(X_reg_train)
X_reg_test  = encode_categoricals(X_reg_test)

# ── Step 2: feature engineering (no loan_amount-derived ratios) ───────────────
def add_features_reg(df):
    df = df.copy()
    df["total_assets"]    = (df["residential_assets_value"]
                             + df["commercial_assets_value"]
                             + df["luxury_assets_value"]
                             + df["bank_asset_value"])
    df["asset_to_income"] = df["total_assets"] / df["income_annum"]
    return df

X_reg_train = add_features_reg(X_reg_train)
X_reg_test  = add_features_reg(X_reg_test)

reg_feature_names = X_reg_train.columns.tolist()
print(f"Regression features ({len(reg_feature_names)}): {reg_feature_names}")

Regression features (12): ['no_of_dependents', 'education', 'self_employed', 'income_annum', 'loan_term', 'cibil_score', 'residential_assets_value', 'commercial_assets_value', 'luxury_assets_value', 'bank_asset_value', 'total_assets', 'asset_to_income']


In [64]:
# ── Step 3: cap outliers — fit on regression training set only ────────────────
REG_CAP_COLS = ["residential_assets_value", "commercial_assets_value", "bank_asset_value"]

reg_cap_bounds = {}
for col in REG_CAP_COLS:
    q1  = X_reg_train[col].quantile(0.25)
    q3  = X_reg_train[col].quantile(0.75)
    iqr = q3 - q1
    reg_cap_bounds[col] = (q1 - 1.5 * iqr, q3 + 1.5 * iqr)
    print(f"  {col}: lower={reg_cap_bounds[col][0]:,.0f}  upper={reg_cap_bounds[col][1]:,.0f}")

for col, (lo, hi) in reg_cap_bounds.items():
    X_reg_train[col] = X_reg_train[col].clip(lower=lo, upper=hi)
    X_reg_test[col]  = X_reg_test[col].clip(lower=lo, upper=hi)

# ── Step 4: log-transform skewed asset columns ────────────────────────────────
REG_SKEWED_COLS = ["residential_assets_value", "commercial_assets_value", "bank_asset_value"]
for col in REG_SKEWED_COLS:
    X_reg_train[col] = np.log1p(np.maximum(X_reg_train[col], 0))
    X_reg_test[col]  = np.log1p(np.maximum(X_reg_test[col],  0))

# ── Step 5: scale features — fit on regression training set only ──────────────
reg_scaler       = StandardScaler()
X_reg_train_proc = reg_scaler.fit_transform(X_reg_train)
X_reg_test_proc  = reg_scaler.transform(X_reg_test)

# ── Log-transform the target ──────────────────────────────────────────────────
y_reg_train_log = np.log1p(y_reg_train)
y_reg_test_log  = np.log1p(y_reg_test)

print(f"\nFinal shape — Train: {X_reg_train_proc.shape}  |  Test: {X_reg_test_proc.shape}")
print(f"NaN check   — Train: {np.isnan(X_reg_train_proc).sum()}  |  Test: {np.isnan(X_reg_test_proc).sum()}")
print(f"\nTarget (log space) — Train mean: {y_reg_train_log.mean():.4f}  std: {y_reg_train_log.std():.4f}")

  residential_assets_value: lower=-11,600,000  upper=25,200,000
  commercial_assets_value: lower=-8,200,000  upper=17,400,000
  bank_asset_value: lower=-5,050,000  upper=14,550,000

Final shape — Train: (2124, 12)  |  Test: (532, 12)
NaN check   — Train: 0  |  Test: 0

Target (log space) — Train mean: 16.2659  std: 0.8762


### Step 4 — Evaluation Helper & Baseline Model Comparison

All models trained on the log-transformed target. Predictions are inverse-transformed (expm1) back to rupees for RMSE, MAE, and MAPE reporting.

In [65]:
def evaluate_regressor(name, model, X_tr, X_te, y_tr_log, y_te_raw):
    model.fit(X_tr, y_tr_log)
    y_pred_log = model.predict(X_te)
    y_pred_raw = np.expm1(y_pred_log)          # back to rupees

    rmse = np.sqrt(mean_squared_error(y_te_raw, y_pred_raw))
    mae  = mean_absolute_error(y_te_raw, y_pred_raw)
    r2   = r2_score(y_te_raw, y_pred_raw)
    mape = np.mean(np.abs((y_te_raw - y_pred_raw) / y_te_raw)) * 100

    return {
        "Model": name,
        "R²":    round(r2,   4),
        "RMSE":  round(rmse, 0),
        "MAE":   round(mae,  0),
        "MAPE%": round(mape, 2),
    }, model, y_pred_raw

reg_models = {
    "Linear Regression":   LinearRegression(),
    "Ridge":               Ridge(alpha=1.0),
    "Lasso":               Lasso(alpha=1.0, max_iter=5000),
    "Decision Tree":       DecisionTreeRegressor(random_state=RANDOM_STATE),
    "Random Forest":       RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE),
    "Gradient Boosting":   GradientBoostingRegressor(n_estimators=100, random_state=RANDOM_STATE),
}

reg_results      = {}
reg_metrics_rows = []

print("Training regression baselines…\n")
for name, model in reg_models.items():
    m, fitted, y_pred = evaluate_regressor(
        name, model,
        X_reg_train_proc, X_reg_test_proc,
        y_reg_train_log, y_reg_test,
    )
    reg_results[name]   = (m, fitted, y_pred)
    reg_metrics_rows.append(m)
    print(f"  {name:<25}  R²={m['R²']:.4f}   RMSE=₹{m['RMSE']:,.0f}   MAPE={m['MAPE%']:.2f}%")

print("\nDone.")

Training regression baselines…

  Linear Regression          R²=0.7800   RMSE=₹4,348,708   MAPE=28.78%
  Ridge                      R²=0.7801   RMSE=₹4,347,153   MAPE=28.78%
  Lasso                      R²=-0.1400   RMSE=₹9,898,530   MAPE=149.31%
  Decision Tree              R²=0.7336   RMSE=₹4,785,473   MAPE=23.92%
  Random Forest              R²=0.8668   RMSE=₹3,384,023   MAPE=18.15%
  Gradient Boosting          R²=0.8761   RMSE=₹3,263,928   MAPE=17.87%

Done.


In [66]:
reg_results_df = (pd.DataFrame(reg_metrics_rows)
                  .set_index("Model")
                  .sort_values("R²", ascending=False))
print(reg_results_df.to_string())

# ── Comparison bar chart ──────────────────────────────────────────────────────
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("R² Score (higher is better)",
                                    "MAPE % (lower is better)"),
                    horizontal_spacing=0.12)

colors = px.colors.qualitative.Plotly
model_order = reg_results_df.index.tolist()

fig.add_trace(go.Bar(
    x=reg_results_df["R²"], y=model_order, orientation="h",
    marker_color=colors[0], text=[f"{v:.4f}" for v in reg_results_df["R²"]],
    textposition="outside", name="R²",
), row=1, col=1)

fig.add_trace(go.Bar(
    x=reg_results_df["MAPE%"], y=model_order, orientation="h",
    marker_color=colors[1], text=[f"{v:.2f}%" for v in reg_results_df["MAPE%"]],
    textposition="outside", name="MAPE%",
), row=1, col=2)

fig.update_layout(
    title    = dict(text="Regression Baseline Comparison",
                    font=dict(size=17, color="#1E3A5F"), x=0.5, xanchor="center"),
    template = "plotly_white", height=380, showlegend=False,
    font=dict(family="Arial, sans-serif", size=11),
    margin=dict(t=80, b=40, l=160, r=80),
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)
fig.show()

                       R²       RMSE        MAE   MAPE%
Model                                                  
Gradient Boosting  0.8761  3263928.0  2435079.0   17.87
Random Forest      0.8668  3384023.0  2505332.0   18.15
Ridge              0.7801  4347153.0  3070315.0   28.78
Linear Regression  0.7800  4348708.0  3070461.0   28.78
Decision Tree      0.7336  4785473.0  3344361.0   23.92
Lasso             -0.1400  9898530.0  8049417.0  149.31


### Step 5 — Hyperparameter Tuning — Random Forest Regressor

Same two-stage strategy as the classifier:
- **Stage 1 — RandomizedSearchCV** over a broad parameter space
- **Stage 2 — GridSearchCV** to zoom in around the Stage 1 winner

In [67]:
reg_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
# KFold for regression (StratifiedKFold is for classification)
from sklearn.model_selection import KFold
reg_cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

reg_param_dist = {
    "n_estimators":      randint(50, 600),
    "max_depth":         [None, 5, 10, 15, 20, 30],
    "min_samples_split": randint(2, 20),
    "min_samples_leaf":  randint(1, 10),
    "max_features":      ["sqrt", "log2", 0.5, None],
    "bootstrap":         [True, False],
}

reg_random_search = RandomizedSearchCV(
    estimator          = RandomForestRegressor(random_state=RANDOM_STATE),
    param_distributions= reg_param_dist,
    n_iter             = 100,
    scoring            = "r2",
    cv                 = reg_cv,
    n_jobs             = -1,
    random_state       = RANDOM_STATE,
    verbose            = 1,
)

print("Stage 1 — RandomizedSearchCV (100 candidates × 5-fold CV)…")
reg_random_search.fit(X_reg_train_proc, y_reg_train_log)

print(f"\nBest R² (CV): {reg_random_search.best_score_:.6f}")
print("Best params  :")
for k, v in reg_random_search.best_params_.items():
    print(f"  {k:<22} {v}")

Stage 1 — RandomizedSearchCV (100 candidates × 5-fold CV)…
Fitting 5 folds for each of 100 candidates, totalling 500 fits

Best R² (CV): 0.945609
Best params  :
  bootstrap              True
  max_depth              5
  max_features           None
  min_samples_leaf       8
  min_samples_split      16
  n_estimators           423


In [68]:
reg_bp = reg_random_search.best_params_

reg_param_grid = {
    "n_estimators":      _neighbors(reg_bp["n_estimators"],     50, lo=50),
    "max_depth":         ([None] if reg_bp["max_depth"] is None
                          else _neighbors(reg_bp["max_depth"], 5, lo=5)),
    "min_samples_split": _neighbors(reg_bp["min_samples_split"], 2, lo=2),
    "min_samples_leaf":  _neighbors(reg_bp["min_samples_leaf"],  1, lo=1),
    "max_features":      [reg_bp["max_features"]],
    "bootstrap":         [reg_bp["bootstrap"]],
}

total_reg_fits = 1
for v in reg_param_grid.values():
    total_reg_fits *= len(v)
print(f"Stage 2 — GridSearchCV ({total_reg_fits} candidates × 5-fold CV)…\n")

reg_grid_search = GridSearchCV(
    estimator = RandomForestRegressor(random_state=RANDOM_STATE),
    param_grid= reg_param_grid,
    scoring   = "r2",
    cv        = reg_cv,
    n_jobs    = -1,
    verbose   = 1,
)
reg_grid_search.fit(X_reg_train_proc, y_reg_train_log)

print(f"\nBest R² (CV): {reg_grid_search.best_score_:.6f}")
print("Best params  :")
for k, v in reg_grid_search.best_params_.items():
    print(f"  {k:<22} {v}")

Stage 2 — GridSearchCV (54 candidates × 5-fold CV)…

Fitting 5 folds for each of 54 candidates, totalling 270 fits

Best R² (CV): 0.945627
Best params  :
  bootstrap              True
  max_depth              5
  max_features           None
  min_samples_leaf       9
  min_samples_split      14
  n_estimators           473


### Step 6 — Final Regression Model: Train, Evaluate & Cross-Validate

In [69]:
reg_best_params = reg_grid_search.best_params_

final_reg = RandomForestRegressor(**reg_best_params, random_state=RANDOM_STATE)
final_reg.fit(X_reg_train_proc, y_reg_train_log)

# ── Test-set evaluation ───────────────────────────────────────────────────────
y_reg_pred_log = final_reg.predict(X_reg_test_proc)
y_reg_pred_raw = np.expm1(y_reg_pred_log)

final_reg_metrics = {
    "R²":    r2_score(y_reg_test,            y_reg_pred_raw),
    "RMSE":  np.sqrt(mean_squared_error(y_reg_test, y_reg_pred_raw)),
    "MAE":   mean_absolute_error(y_reg_test, y_reg_pred_raw),
    "MAPE%": np.mean(np.abs((y_reg_test - y_reg_pred_raw) / y_reg_test)) * 100,
}

# ── 5-fold CV on training data ────────────────────────────────────────────────
cv_r2   = cross_val_score(final_reg, X_reg_train_proc, y_reg_train_log,
                          cv=reg_cv, scoring="r2",                   n_jobs=-1)
cv_rmse = cross_val_score(final_reg, X_reg_train_proc, y_reg_train_log,
                          cv=reg_cv, scoring="neg_root_mean_squared_error", n_jobs=-1)

print("=" * 55)
print("  FINAL RF REGRESSOR — TUNED MODEL")
print("=" * 55)
print("\n  Test-set metrics (predictions in ₹):")
print(f"    R²           {final_reg_metrics['R²']:.6f}")
print(f"    RMSE         ₹{final_reg_metrics['RMSE']:>15,.0f}")
print(f"    MAE          ₹{final_reg_metrics['MAE']:>15,.0f}")
print(f"    MAPE         {final_reg_metrics['MAPE%']:.4f}%")

print("\n  5-fold CV on training data (log-space target):")
print(f"    R²           {cv_r2.mean():.6f}  ± {cv_r2.std():.6f}")
print(f"    RMSE (log)   {-cv_rmse.mean():.6f}  ± {cv_rmse.std():.6f}")

print("\n  Best hyperparameters:")
for k, v in reg_best_params.items():
    print(f"    {k:<22} {v}")
print("=" * 55)

# ── Compare with baseline RF ──────────────────────────────────────────────────
baseline_r2 = reg_results_df.loc["Random Forest", "R²"]
print(f"\n  vs baseline RF:  R² {baseline_r2:.4f} → {final_reg_metrics['R²']:.4f}"
      f"  Δ={final_reg_metrics['R²'] - baseline_r2:+.4f}")

  FINAL RF REGRESSOR — TUNED MODEL

  Test-set metrics (predictions in ₹):
    R²           0.873632
    RMSE         ₹      3,295,682
    MAE          ₹      2,467,503
    MAPE         18.2154%

  5-fold CV on training data (log-space target):
    R²           0.945627  ± 0.006665
    RMSE (log)   0.202585  ± 0.005053

  Best hyperparameters:
    bootstrap              True
    max_depth              5
    max_features           None
    min_samples_leaf       9
    min_samples_split      14
    n_estimators           473

  vs baseline RF:  R² 0.8668 → 0.8736  Δ=+0.0068


### Diagnostic Plots — Actual vs Predicted & Residuals

In [70]:
residuals   = y_reg_test.values - y_reg_pred_raw
pct_error   = residuals / y_reg_test.values * 100
actual_cr   = y_reg_test.values / 1e7      # convert to crores for readability
predicted_cr= y_reg_pred_raw / 1e7

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Actual vs Predicted (₹ Crore)", "Residuals vs Predicted"),
    horizontal_spacing=0.12,
)

# Actual vs Predicted scatter
fig.add_trace(go.Scatter(
    x=actual_cr, y=predicted_cr, mode="markers",
    marker=dict(color="#3B82F6", opacity=0.5, size=5),
    name="Predictions",
), row=1, col=1)

# Perfect prediction line
lo, hi = min(actual_cr.min(), predicted_cr.min()), max(actual_cr.max(), predicted_cr.max())
fig.add_trace(go.Scatter(
    x=[lo, hi], y=[lo, hi], mode="lines",
    line=dict(color="#EF4444", dash="dash", width=1.5),
    name="Perfect fit",
), row=1, col=1)

# Residuals vs Predicted
fig.add_trace(go.Scatter(
    x=predicted_cr, y=pct_error, mode="markers",
    marker=dict(color="#10B981", opacity=0.5, size=5),
    name="% Error",
), row=1, col=2)
fig.add_hline(y=0, line_dash="dash", line_color="#EF4444", line_width=1.5, row=1, col=2)

fig.update_layout(
    title    = dict(text="Final RF Regressor — Diagnostic Plots",
                    font=dict(size=17, color="#1E3A5F"), x=0.5, xanchor="center"),
    template = "plotly_white", height=430, showlegend=False,
    font=dict(family="Arial, sans-serif", size=11),
    margin=dict(t=80, b=60, l=70, r=40),
)
fig.update_xaxes(title_text="Actual (₹ Cr)",    row=1, col=1, showgrid=False)
fig.update_yaxes(title_text="Predicted (₹ Cr)", row=1, col=1, showgrid=False)
fig.update_xaxes(title_text="Predicted (₹ Cr)", row=1, col=2, showgrid=False)
fig.update_yaxes(title_text="% Error",           row=1, col=2, showgrid=False)
fig.show()

### Feature Importance — Regression Model

In [71]:
reg_imp_df = pd.DataFrame({
    "Feature":    reg_feature_names,
    "Importance": final_reg.feature_importances_,
}).sort_values("Importance", ascending=True)

fig = go.Figure(go.Bar(
    x            = reg_imp_df["Importance"],
    y            = reg_imp_df["Feature"],
    orientation  = "h",
    marker       = dict(color=reg_imp_df["Importance"], colorscale="Greens", showscale=False),
    text         = [f"{v:.4f}" for v in reg_imp_df["Importance"]],
    textposition = "outside",
    textfont     = dict(size=11, color="#1F2937"),
))

fig.update_layout(
    title    = dict(text="Final RF Regressor (Tuned) — Feature Importance",
                    font=dict(size=17, color="#1E3A5F"), x=0.5, xanchor="center"),
    template = "plotly_white",
    height   = max(420, len(reg_feature_names) * 35),
    xaxis    = dict(title="Importance Score", showgrid=False),
    yaxis    = dict(showgrid=False),
    font     = dict(family="Arial, sans-serif", size=12),
    margin   = dict(t=80, b=60, l=200, r=120),
)
fig.show()

### Step 7 — End-to-End Pipeline Demo

Chains the two models: **Classifier → Regressor**

Given a raw applicant record, the pipeline:
1. Preprocesses the input
2. Runs the classifier — if rejected, stops and returns the decision
3. If approved, runs the regressor and returns the predicted loan amount in ₹

In [72]:
def preprocess_for_classifier(raw: dict) -> np.ndarray:
    """Apply the classifier preprocessing pipeline to a single applicant dict."""
    row = pd.DataFrame([raw])
    row = encode_categoricals(row)
    row = add_features(row)           # includes loan_amount-derived features
    for col, (lo, hi) in cap_bounds.items():
        row[col] = row[col].clip(lower=lo, upper=hi)
    for col in SKEWED_COLS:
        row[col] = np.log1p(np.maximum(row[col], 0))
    return scaler.transform(row[feature_names])

def preprocess_for_regressor(raw: dict) -> np.ndarray:
    """Apply the regressor preprocessing pipeline to a single applicant dict."""
    row = pd.DataFrame([raw])
    row = encode_categoricals(row)
    row = add_features_reg(row)       # asset_to_income only — no loan_amount leakage
    for col, (lo, hi) in reg_cap_bounds.items():
        row[col] = row[col].clip(lower=lo, upper=hi)
    for col in REG_SKEWED_COLS:
        row[col] = np.log1p(np.maximum(row[col], 0))
    return reg_scaler.transform(row[reg_feature_names])

def predict_loan(applicant: dict) -> dict:
    """
    End-to-end prediction:
      1. Classifier decides approve / reject.
      2. If approved, regressor predicts the loan amount.
    """
    clf_input = preprocess_for_classifier(applicant)
    approved  = bool(final_rf.predict(clf_input)[0])
    prob      = float(final_rf.predict_proba(clf_input)[0][1])

    if not approved:
        return {"decision": "Rejected", "approval_probability": round(prob, 4),
                "predicted_loan_amount": None}

    reg_input    = preprocess_for_regressor(applicant)
    amount_log   = final_reg.predict(reg_input)[0]
    amount_rupees= float(np.expm1(amount_log))

    return {
        "decision":               "Approved",
        "approval_probability":   round(prob, 4),
        "predicted_loan_amount":  round(amount_rupees, 0),
        "predicted_loan_amount_cr": round(amount_rupees / 1e7, 2),
    }


# ── Demo: two sample applicants ───────────────────────────────────────────────
applicant_approved = {
    "no_of_dependents": 2,  "education": "Graduate",  "self_employed": "No",
    "income_annum": 9600000,  "loan_amount": 29900000,  "loan_term": 12,
    "cibil_score": 778,  "residential_assets_value": 2400000,
    "commercial_assets_value": 17600000,  "luxury_assets_value": 22700000,
    "bank_asset_value": 8000000,
}

applicant_rejected = {
    "no_of_dependents": 0,  "education": "Not Graduate",  "self_employed": "Yes",
    "income_annum": 4100000,  "loan_amount": 12200000,  "loan_term": 8,
    "cibil_score": 417,  "residential_assets_value": 2700000,
    "commercial_assets_value": 2200000,  "luxury_assets_value": 8800000,
    "bank_asset_value": 3300000,
}

for label, applicant in [("Applicant A (high CIBIL)", applicant_approved),
                          ("Applicant B (low CIBIL)",  applicant_rejected)]:
    result = predict_loan(applicant)
    print(f"\n{label}")
    print(f"  Decision             : {result['decision']}")
    print(f"  Approval probability : {result['approval_probability']:.4f}")
    if result["predicted_loan_amount"]:
        print(f"  Predicted loan amount: ₹{result['predicted_loan_amount']:>15,.0f}"
              f"  ({result['predicted_loan_amount_cr']} Cr)")


Applicant A (high CIBIL)
  Decision             : Approved
  Approval probability : 1.0000
  Predicted loan amount: ₹     29,647,542  (2.96 Cr)

Applicant B (low CIBIL)
  Decision             : Rejected
  Approval probability : 0.0000


### Step 8 — Save Regression Model & Artifacts

In [73]:
# Save regression model as pickle
with open(models_dir / "random_forest_regressor_final.pkl", "wb") as f:
    pickle.dump(final_reg, f)

# Save regression scaler
joblib.dump(reg_scaler, models_dir / "reg_scaler.joblib")

# Save regression cap bounds
reg_cap_bounds_serialisable = {k: list(v) for k, v in reg_cap_bounds.items()}
with open(models_dir / "reg_cap_bounds.json", "w") as f:
    json.dump(reg_cap_bounds_serialisable, f, indent=2)

# Save regression model metadata
reg_metadata = {
    "feature_names":        reg_feature_names,
    "target":               "loan_amount (₹)",
    "target_transform":     "log1p during training; expm1 to get predictions in ₹",
    "training_subset":      "approved loans only (loan_status == 1)",
    "best_hyperparams":     {k: (v if v is not None else None)
                             for k, v in reg_best_params.items()},
    "cv_r2_mean":           round(float(cv_r2.mean()),    6),
    "cv_r2_std":            round(float(cv_r2.std()),     6),
    "test_r2":              round(final_reg_metrics["R²"],    6),
    "test_rmse_rupees":     round(final_reg_metrics["RMSE"],  0),
    "test_mae_rupees":      round(final_reg_metrics["MAE"],   0),
    "test_mape_pct":        round(final_reg_metrics["MAPE%"], 4),
}
with open(models_dir / "reg_model_metadata.json", "w") as f:
    json.dump(reg_metadata, f, indent=2)

print("Saved to", models_dir.resolve())
for p in sorted(models_dir.iterdir()):
    print(f"  {p.name}  ({p.stat().st_size / 1024:.1f} KB)")

# Round-trip verification
with open(models_dir / "random_forest_regressor_final.pkl", "rb") as f:
    loaded_reg = pickle.load(f)
sample_pred = np.expm1(loaded_reg.predict(X_reg_test_proc[:3]))
print(f"\nPickle verification — sample predictions (₹): {[f'{v:,.0f}' for v in sample_pred]}")

Saved to /Users/dhruv/projects/Intelligent End-to-End Loan Approval & Valuation System/models
  cap_bounds.json  (0.2 KB)
  model_metadata.json  (0.6 KB)
  random_forest_final.joblib  (572.3 KB)
  random_forest_final.pkl  (533.7 KB)
  random_forest_regressor_final.pkl  (2080.3 KB)
  reg_cap_bounds.json  (0.2 KB)
  reg_model_metadata.json  (0.8 KB)
  reg_scaler.joblib  (1.3 KB)
  scaler.joblib  (1.4 KB)

Pickle verification — sample predictions (₹): ['11,673,693', '19,484,847', '6,733,435']
